# Tuotrial: Buffer capture using SR860

## Imports

In [ ]:
import numpy as np
from scipy.io import savemat
import sys, yaml, logging, time, json, datetime as dt
from pathlib import Path

import qcodes as qc
from qcodes.station import Station
from qcodes.parameters import Parameter
from qcodes.dataset import (
    Measurement,
    initialise_or_create_database_at,
    load_or_create_experiment
)
from qcodes.instrument import Instrument
from qcodes.instrument_drivers.stanford_research import SR860

## Setup logging

In [ ]:
log = logging.getLogger("tutorial")
log.setLevel(logging.INFO)
log.propagate = False
for h in list(log.handlers):          # clear stale handlers on re-run
    log.removeHandler(h)
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-7s %(message)s", "%H:%M:%S"))
log.addHandler(h)

qc.logger.start_all_logging()
log.info("QCoDeS log files in: %s", qc.logger.get_log_file_name())

## Read Configuration

In [ ]:
CONFIG_PATH = Path("config.yaml").resolve()
ROOT = CONFIG_PATH.parent
config_text = CONFIG_PATH.read_text()
cfg = yaml.safe_load(config_text)

log.info("loaded config: %s", 'path to config')

## Database and Experiment

In [ ]:
db_path = (ROOT / cfg["database"]).resolve()
db_path.parent.mkdir(parents=True, exist_ok=True)

initialise_or_create_database_at(db_path)
log.info("Database: %s (%.1f MB)", db_path, db_path.stat().st_size / 1e6)

exp = load_or_create_experiment(
    experiment_name = cfg["experiment_name"],
    sample_name = cfg['sample_name']
)
log.info("Experiment #%d '%s' on sample '%s' - %d run(s) so far", exp.exp_id, exp.name, exp.sample_name, len(exp.data_sets()))

## Instruments and Station

In [ ]:
Instrument.close_all()
station = Station()

lck_cfg = cfg['instruments']['lockin']

t0 = time.perf_counter()
lockin = SR860("lockin", lck_cfg['address'])
log.info("Connected to %s at %s in %.2f s",
         lockin.IDN(), lck_cfg['address'], time.perf_counter()-t0)

for pname, requested in lck_cfg.get("settings", {}).items():
    param = lockin.parameters[pname]
    param(requested)
    time.sleep(0.5)
    actual = param()
    flag = "" if actual == requested else "  <-- DIFFERS from requested %r" % requested
    log.info("%-16s = %-10r %s%s", pname, actual, param.unit, flag)

station.add_component(lockin)
log.info("Station components: %s", list(station.components))

## Buffer settings

In [ ]:
bcfg = cfg["buffer"]
mcfg = cfg["measurement"]

buf = lockin.buffer
buf.capture_config(bcfg["capture_config"])
time.sleep(0.5)

rate = buf.capture_rate()
channels = bcfg["capture_config"].split(",")
n_samples = int(mcfg["duration"] * rate)
n_bytes = n_samples * len(channels) * 4

buf.set_capture_length_to_fit_samples(n_samples)

log.info("%d samples x %d channels = %.1f kB (buffer max 4096 kB)", 
         n_samples, len(channels), n_bytes/1024)

## Register parameters

In [ ]:
meas = Measurement(
    exp = exp,
    station = station,
    name = "SR860_buffer_capture"
)
meas.write_period = 1.0

meas.register_custom_parameter("t", 
                                label = "Time",
                                unit = "s",
                                paramtype = "array"
                                )
for ch in channels:
    meas.register_custom_parameter(ch, 
                                   label = f"Lock-in {ch}",
                                   unit = "deg" if ch == "P" else "V",
                                   setpoints = ("t",),
                                   paramtype = "array"
                                   )

log.info("Registered %d columns:", len(meas.parameters))

## Measurement

In [ ]:
with meas.run() as datasaver:
    ds = datasaver.dataset
    log.info("Run #%d started: %d samples at %.1f Hz (%.1f s)", 
             ds.run_id, n_samples, rate, n_samples / rate)

    buf.stop_capture()
    time.sleep(0.5)
    buf. start_capture("ONE", "IMM")

    t_start = time.perf_counter()
    timeout = n_samples / rate * 2 + 5.0

    while buf.count_capture_bytes() < n_bytes:
        if time.perf_counter() - t_start > timeout:
            buf.stop_capture()
            raise TimeoutError("Timeout waiting for buffer capture to complete")
        time.sleep(0.5)

    buf.stop_capture()
    t_total = time.perf_counter() - t_start
    
    bytes_kb = buf.count_capture_bytes() / 1024
    prog_kb = buf.count_capture_kilobytes()
    log.info("buffer: captured %.1f kB (CAPTUREBYTES), %s kB (CAPTUREPROG), requested %d samples = %.1f kB",
             bytes_kb, prog_kb, n_samples, n_bytes / 1024)

    #n_avail = int(min(n_bytes, bytes_kb * 1024) // (4 * len(channels)))
    #if n_avail < n_samples:
     #   log.warning("only %d of %d samples available; reading what is there", n_avail, n_samples)
    data = buf.get_capture_data(n_samples)
    
    n_got = len(data[channels[0]])
    t = np.arange(n_got) / rate

    datasaver.add_result(
        ("t", t),
        *[(ch, data[ch]) for ch in channels]
    )
    ds.add_metadata("capture_rate_hz", rate)
    
log.info("Run #%d finished: %d points in %.2f s (%.1f Hz)", ds.run_id, n_got, t_total, n_got/t_total)

## Matlab data format

In [ ]:
export_dir = (ROOT / cfg["export_dir"]).resolve()
export_dir.mkdir(parents=True, exist_ok=True)

df = ds.to_pandas_dataframe().reset_index()
stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
mat_path = export_dir / f"run{ds.run_id:04}_{ds.guid[:8]}_{stamp}.mat"

mat = {col: df[col].to_numpy() for col in df.columns}
mat.update({
    "run_id": ds.run_id,
    "guid": ds.guid,
    "experiment": exp.name,
    "sample": exp.sample_name,
    "config_yaml": config_text,
    "snapshot_json": json.dumps(ds.snapshot),
    "units": {"t": "s", **{ch: ("deg" if ch == "T" else "V") for ch in channels}},
})

savemat(mat_path, mat, do_compression=True)
log.info("MATLAB file: %s (%.1f kB, %d rows)",
         mat_path.name, mat_path.stat().st_size / 1e3, len(df))